In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor,
    HistGradientBoostingRegressor
)
from sklearn.neighbors import KNeighborsRegressor

In [2]:
df = pd.read_csv("data/clean_data.csv")

In [3]:
features = [
    "apartment_type",
    "metro_station",
    "minutes_to_metro",
    "rooms",
    "area",
    "living_area",
    "kitchen_area",
    "floor",
    "total_floors",
    "renovation"
]

numeric_features = [
    "minutes_to_metro",
    "rooms",
    "area",
    "living_area",
    "kitchen_area",
    "floor",
    "total_floors"
]

categorical_features = [
    "apartment_type",
    "metro_station",
    "renovation"
]

target = "price"

X = df[features]
y = df[target]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (14184, 10)
Test: (3547, 10)


In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [6]:
models = {
    "Linear Regression": LinearRegression(),

    "ElasticNet": ElasticNet(
        alpha=0.001,
        l1_ratio=0.5,
        max_iter=10_000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42,
        max_depth=12,
        min_samples_leaf=5
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3
    ),
}

In [7]:
def train_and_evaluate_model(model_name, model):
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    metrics = {
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

    return pipeline, metrics

In [8]:
results = []
trained_models = {}

for model_name, model in models.items():
    print(f"Обучается модель: {model_name}")

    try:
        trained_pipeline, metrics = train_and_evaluate_model(model_name, model)

        results.append(metrics)
        trained_models[model_name] = trained_pipeline

        print(
            f"{model_name}: "
            f"MAE={metrics['MAE']:,.0f}, "
            f"RMSE={metrics['RMSE']:,.0f}, "
            f"R2={metrics['R2']:.4f}"
        )

    except Exception as error:
        print(f"{model_name} не обучилась. Ошибка: {error}")

print()

Обучается модель: Linear Regression
Linear Regression: MAE=6,862,349, RMSE=12,301,727, R2=0.8323
Обучается модель: ElasticNet
ElasticNet: MAE=7,199,760, RMSE=12,605,382, R2=0.8239
Обучается модель: Decision Tree
Decision Tree: MAE=5,466,354, RMSE=14,195,657, R2=0.7767
Обучается модель: Random Forest
Random Forest: MAE=4,369,536, RMSE=11,381,837, R2=0.8565
Обучается модель: Extra Trees
Extra Trees: MAE=4,179,146, RMSE=11,036,731, R2=0.8650
Обучается модель: Gradient Boosting
Gradient Boosting: MAE=5,369,886, RMSE=11,383,393, R2=0.8564



In [9]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("MAE").reset_index(drop=True)

results_df

,model,MAE,RMSE,R2
0,Extra Trees,4.179146e+06,1.103673e+07,0.865026
1,Random Forest,4.369536e+06,1.138184e+07,0.856453
2,Gradient Boosting,5.369886e+06,1.138339e+07,0.856414
3,Decision Tree,5.466354e+06,1.419566e+07,0.776704
4,Linear Regression,6.862349e+06,1.230173e+07,0.832312
5,ElasticNet,7.199760e+06,1.260538e+07,0.823932


In [10]:
best_model_name = results_df.loc[0, "model"]
best_model = trained_models[best_model_name]

print("Лучшая модель:", best_model_name)
print("MAE:", round(results_df.loc[0, "MAE"]))
print("RMSE:", round(results_df.loc[0, "RMSE"]))
print("R2:", round(results_df.loc[0, "R2"], 4))

Лучшая модель: Extra Trees
MAE: 4179146
RMSE: 11036731
R2: 0.865


In [11]:
joblib.dump(best_model, "models/best_model.pkl")

['models/best_model.pkl']

In [12]:
results_df.to_csv("models/metrics.csv", index=False)

In [13]:
metadata = {
    "apartment_types": sorted(df["apartment_type"].dropna().unique().tolist()),
    "metro_stations": sorted(df["metro_station"].dropna().unique().tolist()),
    "renovations": sorted(df["renovation"].dropna().unique().tolist()),

    "features": features,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "target": target,

    "best_model": best_model_name,

    "metrics": {
        "MAE": float(results_df.loc[0, "MAE"]),
        "RMSE": float(results_df.loc[0, "RMSE"]),
        "R2": float(results_df.loc[0, "R2"])
    }
}

joblib.dump(metadata, "models/metadata.pkl")

['models/metadata.pkl']